# 01 — Consolidação da base e filtro da instituição

Este notebook realiza apenas a construção do **dataset inicial da pesquisa**, preservando os dados originais tanto quanto possível.

Etapas executadas:
1. Localiza os arquivos mensais no padrão `finalizadas_AAAA-MM.csv`.
2. Lê e concatena os 48 arquivos.
3. Filtra as observações em que `Nome Fantasia = Banco do Brasil`.
4. Cria a variável `AAAA_MM` a partir de `Data Finalização`.
5. Salva o dataset consolidado em CSV.

**Importante:** neste estágio não são realizados tratamentos de valores ausentes, recodificações, exclusões de variáveis ou alterações na variável-alvo.

In [1]:
# Importação das bibliotecas necessárias
from pathlib import Path
import pandas as pd


## 1. Definição dos diretórios

Os arquivos mensais estão armazenados em `C:\\UnB\\script_md\\Base de dados`.
O dataset consolidado será salvo em uma subpasta chamada `processados`.

In [4]:
# Diretório onde estão os 48 arquivos mensais
BASE_DIR = Path(r"C:\UnB\script_md\Base de dados")

# Diretório onde será salvo o dataset consolidado
OUTPUT_DIR = BASE_DIR / "processados"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Arquivo de saída
OUTPUT_FILE = OUTPUT_DIR / "dataset.csv"

print(f"Diretório de entrada: {BASE_DIR}")
print(f"Arquivo de saída:     {OUTPUT_FILE}")


Diretório de entrada: C:\UnB\script_md\Base de dados
Arquivo de saída:     C:\UnB\script_md\Base de dados\processados\dataset.csv


## 2. Localização dos arquivos mensais

O padrão esperado é `finalizadas_AAAA-MM.csv`. Como o período analisado possui quatro anos, são esperados **48 arquivos**.

In [5]:
# Localiza apenas os arquivos que seguem o padrão definido para a pesquisa
arquivos = sorted(BASE_DIR.glob("finalizadas_????-??.csv"))

print(f"Quantidade de arquivos encontrados: {len(arquivos)}")

# Verificação de segurança: a pesquisa espera exatamente 48 arquivos mensais
assert len(arquivos) == 48, (
    f"Foram encontrados {len(arquivos)} arquivos. "
    "Verifique se os 48 arquivos mensais estão na pasta e seguem o padrão "
    "finalizadas_AAAA-MM.csv."
)

# Exibe o primeiro e o último arquivo apenas para conferência
print(f"Primeiro arquivo: {arquivos[0].name}")
print(f"Último arquivo:   {arquivos[-1].name}")


Quantidade de arquivos encontrados: 48
Primeiro arquivo: finalizadas_2022-08.csv
Último arquivo:   finalizadas_2026-07.csv


## 3. Função de leitura dos CSVs

Os arquivos do Consumidor.gov.br utilizam separador por ponto e vírgula (`;`).
A função abaixo tenta inicialmente `utf-8-sig` e, caso necessário, utiliza `latin-1`, sem alterar os dados.

In [6]:
def ler_csv(caminho):
    """Lê um arquivo CSV utilizando os encodings mais comuns da base."""
    try:
        return pd.read_csv(caminho, sep=";", encoding="utf-8-sig", low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(caminho, sep=";", encoding="latin-1", low_memory=False)


## 4. Consolidação dos 48 arquivos

Nesta etapa os arquivos são apenas lidos e concatenados verticalmente. Nenhuma linha é removida e nenhuma variável é transformada.

In [7]:
# Lista que armazenará temporariamente cada arquivo mensal
bases_mensais = []

for arquivo in arquivos:
    df_mes = ler_csv(arquivo)
    bases_mensais.append(df_mes)

# Consolida todos os meses em um único DataFrame
df = pd.concat(bases_mensais, ignore_index=True)

print(f"Dimensão da base consolidada: {df.shape[0]:,} linhas x {df.shape[1]} colunas")


Dimensão da base consolidada: 7,960,563 linhas x 21 colunas


## 5. Identificação segura das colunas utilizadas

Para evitar problemas causados por espaços acidentais nos cabeçalhos, localizamos as colunas por uma versão normalizada de seus nomes, **sem renomear as colunas do dataset original**.

In [8]:
def localizar_coluna(dataframe, nome_esperado):
    """Localiza uma coluna ignorando espaços duplicados nas extremidades ou entre palavras."""
    alvo = " ".join(nome_esperado.split())

    for coluna in dataframe.columns:
        coluna_normalizada = " ".join(str(coluna).split())
        if coluna_normalizada == alvo:
            return coluna

    raise KeyError(f"Coluna '{nome_esperado}' não encontrada na base.")

COL_EMPRESA = localizar_coluna(df, "Nome Fantasia")
COL_DATA_FINALIZACAO = localizar_coluna(df, "Data Finalização")

print(f"Coluna da instituição: {COL_EMPRESA!r}")
print(f"Coluna da data:        {COL_DATA_FINALIZACAO!r}")


Coluna da instituição: 'Nome Fantasia'
Coluna da data:        'Data Finalização'


## 6. Filtro da instituição analisada

A pesquisa utiliza somente as reclamações cuja variável `Nome Fantasia` corresponde a `Banco do Brasil`.

O filtro remove apenas espaços nas extremidades **para fins de comparação**, preservando o conteúdo armazenado na coluna.

In [9]:
# Quantidade de registros antes do filtro
n_total = len(df)

# Filtra somente a instituição definida para a pesquisa
mascara_bb = df[COL_EMPRESA].astype("string").str.strip().eq("Banco do Brasil")
df_bb = df.loc[mascara_bb].copy()

print(f"Registros na base completa:       {n_total:,}")
print(f"Registros do Banco do Brasil:     {len(df_bb):,}")
print(f"Percentual do total consolidado:  {len(df_bb) / n_total:.2%}")


Registros na base completa:       7,960,563
Registros do Banco do Brasil:     209,870
Percentual do total consolidado:  2.64%


## 7. Criação da variável temporal `AAAA_MM`

A nova variável é derivada diretamente de `Data Finalização` e representa o ano e o mês da reclamação finalizada.

Exemplo: `15/08/2026` → `2026_08`.

Nenhuma observação é excluída caso a data não possa ser interpretada; nesses casos `AAAA_MM` permanecerá ausente para posterior auditoria.

In [10]:
# Converte a data apenas em uma série auxiliar, preservando a coluna original
data_finalizacao = pd.to_datetime(
    df_bb[COL_DATA_FINALIZACAO],
    dayfirst=True,
    errors="coerce"
)

# Cria a variável no formato AAAA_MM
df_bb["AAAA_MM"] = data_finalizacao.dt.strftime("%Y_%m")

# Auditoria da conversão
datas_invalidas = data_finalizacao.isna().sum()

print(f"Datas não interpretadas: {datas_invalidas:,}")
print()
print("Primeiros períodos encontrados:")
print(df_bb["AAAA_MM"].value_counts(dropna=False).sort_index().head(12))


Datas não interpretadas: 125,251

Primeiros períodos encontrados:
AAAA_MM
2022_01    181
2022_02    184
2022_03     92
2022_04    126
2022_05    177
2022_06     79
2022_07    163
2022_08    173
2022_09    172
2022_10    124
2022_11    106
2022_12    167
Name: count, dtype: int64


## 8. Verificações finais

Antes de salvar o dataset, são realizadas verificações simples para confirmar que:
- todos os registros pertencem à instituição definida;
- a nova coluna `AAAA_MM` foi criada;
- as demais colunas foram preservadas.

In [11]:
# Confirma que o filtro contém somente Banco do Brasil
empresas_resultantes = (
    df_bb[COL_EMPRESA]
    .astype("string")
    .str.strip()
    .dropna()
    .unique()
)

assert set(empresas_resultantes) == {"Banco do Brasil"}, (
    f"Foram encontrados valores inesperados em Nome Fantasia: {empresas_resultantes}"
)

assert "AAAA_MM" in df_bb.columns

print(f"Dimensão final: {df_bb.shape[0]:,} linhas x {df_bb.shape[1]} colunas")
print(f"Período inicial: {df_bb['AAAA_MM'].dropna().min()}")
print(f"Período final:   {df_bb['AAAA_MM'].dropna().max()}")

# Visualização de uma amostra sem alterar os dados
df_bb.head()


Dimensão final: 209,870 linhas x 22 colunas
Período inicial: 2022_01
Período final:   2026_12


,Região,UF,Cidade,Sexo,Faixa Etária,Data Finalização,Tempo Resposta,Nome Fantasia,Segmento de Mercado,Área,...,Problema,Como Comprou Contratou,Procurou Empresa,Respondida,Situação,Avaliação Reclamação,Nota do Consumidor,Interação com Judiciario,Último Complemento Consumidor,AAAA_MM
69,N,AM,Manaus,F,entre 51 a 60 anos,2022-08-04,10.0,Banco do Brasil,"Bancos, Financeiras e Administradoras de Cartão",Serviços Financeiros,...,Não entrega do contrato ou documentação relaci...,Loja física,S,S,Finalizada avaliada,Não Resolvida,2.0,NaN,NaN,2022_04
164,SE,SP,São Paulo,M,entre 61 a 70 anos,2022-08-05,10.0,Banco do Brasil,"Bancos, Financeiras e Administradoras de Cartão",Serviços Financeiros,...,Alteração / rescisão de contrato sem solicitaç...,Não comprei / contratei,S,S,Finalizada avaliada,Não Resolvida,2.0,NaN,NaN,2022_05
235,SE,SP,Jundiaí,M,entre 51 a 60 anos,2022-08-11,10.0,Banco do Brasil,"Bancos, Financeiras e Administradoras de Cartão",Serviços Financeiros,...,Cobrança por serviço/produto não contratado / ...,Não comprei / contratei,S,S,Finalizada não avaliada,Não Avaliada,NaN,NaN,NaN,2022_11
306,SE,RJ,Rio de Janeiro,M,entre 61 a 70 anos,2022-08-04,9.0,Banco do Brasil,"Bancos, Financeiras e Administradoras de Cartão",Serviços Financeiros,...,Cobrança por serviço/produto não contratado / ...,SMS / Mensagem de texto,S,S,Finalizada não avaliada,Não Avaliada,NaN,NaN,NaN,2022_04
322,SE,ES,Vitória,F,entre 41 a 50 anos,2022-08-08,8.0,Banco do Brasil,"Bancos, Financeiras e Administradoras de Cartão",Serviços Financeiros,...,Cobrança indevida / abusiva para alterar ou ca...,Telefone,N,S,Finalizada não avaliada,Não Avaliada,NaN,NaN,NaN,2022_08


## 9. Salvamento do dataset puro

O arquivo é salvo com:
- separador `;`;
- codificação `UTF-8 com BOM` (`utf-8-sig`), adequada para abertura no Excel em ambiente Windows;
- sem índice do pandas.


In [12]:
df_bb.to_csv(
    OUTPUT_FILE,
    sep=";",
    encoding="utf-8-sig",
    index=False
)

print("Dataset salvo com sucesso.")
print(f"Caminho: {OUTPUT_FILE}")
print(f"Registros: {len(df_bb):,}")
print(f"Colunas:   {df_bb.shape[1]}")


Dataset salvo com sucesso.
Caminho: C:\UnB\script_md\Base de dados\processados\dataset.csv
Registros: 209,870
Colunas:   22


In [13]:
df_bb

,Região,UF,Cidade,Sexo,Faixa Etária,Data Finalização,Tempo Resposta,Nome Fantasia,Segmento de Mercado,Área,...,Problema,Como Comprou Contratou,Procurou Empresa,Respondida,Situação,Avaliação Reclamação,Nota do Consumidor,Interação com Judiciario,Último Complemento Consumidor,AAAA_MM
69,N,AM,Manaus,F,entre 51 a 60 anos,2022-08-04,10.0,Banco do Brasil,"Bancos, Financeiras e Administradoras de Cartão",Serviços Financeiros,...,Não entrega do contrato ou documentação relaci...,Loja física,S,S,Finalizada avaliada,Não Resolvida,2.0,NaN,NaN,2022_04
164,SE,SP,São Paulo,M,entre 61 a 70 anos,2022-08-05,10.0,Banco do Brasil,"Bancos, Financeiras e Administradoras de Cartão",Serviços Financeiros,...,Alteração / rescisão de contrato sem solicitaç...,Não comprei / contratei,S,S,Finalizada avaliada,Não Resolvida,2.0,NaN,NaN,2022_05
235,SE,SP,Jundiaí,M,entre 51 a 60 anos,2022-08-11,10.0,Banco do Brasil,"Bancos, Financeiras e Administradoras de Cartão",Serviços Financeiros,...,Cobrança por serviço/produto não contratado / ...,Não comprei / contratei,S,S,Finalizada não avaliada,Não Avaliada,NaN,NaN,NaN,2022_11
306,SE,RJ,Rio de Janeiro,M,entre 61 a 70 anos,2022-08-04,9.0,Banco do Brasil,"Bancos, Financeiras e Administradoras de Cartão",Serviços Financeiros,...,Cobrança por serviço/produto não contratado / ...,SMS / Mensagem de texto,S,S,Finalizada não avaliada,Não Avaliada,NaN,NaN,NaN,2022_04
322,SE,ES,Vitória,F,entre 41 a 50 anos,2022-08-08,8.0,Banco do Brasil,"Bancos, Financeiras e Administradoras de Cartão",Serviços Financeiros,...,Cobrança indevida / abusiva para alterar ou ca...,Telefone,N,S,Finalizada não avaliada,Não Avaliada,NaN,NaN,NaN,2022_08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7960418,SE,SP,Piracicaba,M,entre 61 a 70 anos,2026-07-31,1.0,Banco do Brasil,"Bancos, Financeiras e Administradoras de Cartão",Serviços Financeiros,...,Negativação indevida referente à pagamento já ...,Loja física,N,S,Finalizada avaliada,Resolvida,4.0,NaN,NaN,NaN
7960425,NE,CE,Groairas,F,entre 41 a 50 anos,2026-07-31,1.0,Banco do Brasil,"Bancos, Financeiras e Administradoras de Cartão",Serviços Financeiros,...,Renegociação / parcelamento de dívida,Loja física,S,S,Finalizada avaliada,Resolvida,4.0,NaN,NaN,NaN
7960449,SE,SP,Santos,M,entre 31 a 40 anos,2026-07-31,0.0,Banco do Brasil,"Bancos, Financeiras e Administradoras de Cartão",Serviços Financeiros,...,Dificuldade de contato / acesso a outros canai...,Loja física,N,S,Finalizada avaliada,Resolvida,5.0,NaN,NaN,NaN
7960461,SE,SP,Reginópolis,F,entre 21 a 30 anos,2026-07-31,0.0,Banco do Brasil,"Bancos, Financeiras e Administradoras de Cartão",Serviços Financeiros,...,Negativação indevida referente à pagamento já ...,Internet,N,S,Finalizada avaliada,Não Resolvida,5.0,NaN,NaN,NaN


## Resultado desta etapa

O arquivo `dataset.csv` constitui a primeira base consolidada do estudo.

Neste ponto, a única seleção realizada foi a instituição `Banco do Brasil`, e a única variável acrescentada foi `AAAA_MM`, derivada de `Data Finalização`.

O tratamento da variável-alvo, valores ausentes, possíveis inconsistências e variáveis com risco de *data leakage* será realizado somente nas próximas etapas.